# Perfiles clínicos

Este notebook recorre los conceptos del hito 4 de EDsim con las funciones del paquete
`edsim`: el paciente con perfil (edad, motivo, gravedad y triaje con error), los
números aleatorios comunes en escenarios clínicos, el nivel 1 en la puerta, las pruebas y
las duraciones, los tres relojes hasta el médico, la calibración del centro contra datos
españoles y el coste del infratriaje.

Toda la lógica vive en el paquete y está testeada; aquí solo se llama a sus funciones, se
dibuja y se explica.

## 1. Cómo usar este notebook

- **Kernel:** en Cursor, pulsa *Select Kernel* y elige el entorno de `.venv` del proyecto.
  Si no aparece, ejecuta antes `uv sync --all-packages` en la terminal.
- **Ejecutar:** usa *Restart & Run All*. El notebook funciona de principio a fin sin
  depender del orden en que ejecutaste las celdas a mano.
- **Semillas:** cada sección fija su semilla en una constante visible. Con la misma semilla
  el resultado es idéntico; cámbiala para ver otra muestra del azar.
- **Duración:** en modo normal tarda unos 6 minutos; casi todo es la calibración
  de la sección 9 (nueve evaluaciones de 12 réplicas) y las dos ejecuciones de 91 días (secciones 6, 8, 10 y 11). Para ir más
  rápido, baja `DIAS` en la celda de parámetros o usa menos meses en `FECHAS_CAL`.
- **Modo rápido:** con la variable de entorno `EDSIM_NB_RAPIDO=1` los días, las réplicas
  y los pacientes se reducen y el notebook tarda menos de un minuto. Es lo que hace el CI
  (`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`) para comprobar que se ejecuta
  sin errores; los números de ese modo no sirven para sacar conclusiones. **Las cifras que
  cita el texto son las de la ejecución normal.**
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` de pre-commit borra
  resultados e imágenes antes de guardar el notebook en git. Vuelve a ejecutarlo para verlas.
- **El modelo** es el clínico del hito 4: los perfiles de `config/perfiles.toml`, las
  llegadas de `config/llegadas.toml`, el centro calibrado de `config/centro.toml`
  (enfermería de triaje [3, 2, 1] y médicos [12, 11, 4], calibrado en la tarea 4.6c con
  la ventana anual) y el cumplidor de `config/centro_cumplidor.toml` (médicos
  [14, 14, 6]). El notebook 03 sigue con la
  plantilla del hito 3. Se ejecuta desde la carpeta `notebooks/`, de ahí las rutas
  `../config/`.
- **Memoria:** cada ejecución larga se reduce a su tabla de episodios y su ground truth
  nada más simularla, y sus eventos se liberan. La última celda imprime el pico de memoria
  del kernel.

In [ ]:
import dataclasses
import os
import resource
import time
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from edsim.arrivals.profile import load_profile
from edsim.arrivals.rate import MINUTES_PER_DAY
from edsim.arrivals.report import WEEKDAYS
from edsim.calibration import sas
from edsim.calibration.evaluate import (
    CHECK_SEED,
    DRAIN_DAYS,
    FIT_SEED,
    choose_staffing,
    evaluate_staffings,
    replica_values,
)
from edsim.calibration.staffing import DOCTORS, NURSES, knob_loads, rule_staffings
from edsim.calibration.targets import load_targets
from edsim.calibration.window import ANNUAL, window_start_dates
from edsim.engine.analysis import episode_table, paired_waits
from edsim.engine.center import load_center
from edsim.engine.engine import simulate_ed
from edsim.engine.events import MemorySink, to_json
from edsim.engine.report import MUTED_COLOR, OBSERVED_COLOR, SECOND_COLOR, THEORY_COLOR
from edsim.engine.targets import (
    CLOCKS,
    MANCHESTER_MAX_WAIT,
    classified_late,
    clock_quantiles,
    manchester_compliance,
    true_level_compliance,
    true_level_waits,
)
from edsim.lognormal import lognormal_pdf
from edsim.profiles.config import REASON_KEYS, TEST_TYPES, load_profiles
from edsim.profiles.context import ArrivalContext
from edsim.profiles.coupling import (
    gumbel_change_share,
    inverse_cdf_change_share,
    reason_shock_changes,
    simulate_changes,
    simulate_reason_shock,
    total_variation,
)
from edsim.profiles.derivation import with_base_weights, with_target_mix
from edsim.profiles.durations import duration_model, duration_params
from edsim.profiles.mixing import (
    catalog_shares,
    global_reason_shares,
    reason_probabilities,
    reason_weights,
)
from edsim.profiles.population import age_range
from edsim.profiles.severity import (
    UNDER,
    SeverityMixture,
    assigned_shares,
    severity_mixture,
    triage_error_rates,
    triage_matrix,
)
from edsim.profiles.validation import share_interval
from edsim.profiles.workup import order_probability, workup_model

INICIO = time.perf_counter()

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) todo se reduce para que el notebook tarde poco.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

CONFIG = Path("../config")  # el notebook se ejecuta desde notebooks/
CENTRO = load_center(CONFIG / "centro.toml")  # calibrado: enfermería [3, 2, 1], médicos [12, 11, 4]
CUMPLIDOR = load_center(CONFIG / "centro_cumplidor.toml")  # médicos [14, 14, 6]
PERFILES = load_profiles(CONFIG / "perfiles.toml")
LLEGADAS = load_profile(CONFIG / "llegadas.toml")
OBJETIVOS = load_targets(CONFIG / "objetivos.toml")

CAL = 1 if RAPIDO else 2  # días de calentamiento
DIAS = 5 if RAPIDO else 91  # ejecución principal: se mide a los llegados en [CAL, DIAS)
# Sección 9, con la ventana anual: una réplica por mes, desde el primer lunes de cada mes (12
# réplicas de 14 días con 2 de calentamiento); en modo rápido, solo enero y julio, de 3 días.
FECHAS_CAL = window_start_dates(ANNUAL, LLEGADAS)[:: 6 if RAPIDO else 1]
REPLICAS_CAL, DIAS_CAL, CAL_CAL = len(FECHAS_CAL), (3 if RAPIDO else 14), (1 if RAPIDO else 2)
PACIENTES_GRIPE = 5_000 if RAPIDO else None  # sección 4; None es un año (109 542)
PACIENTES_CONTEXTO = 2_000 if RAPIDO else 20_000  # sección 4, un contexto
DIAS_EXP = 5 if RAPIDO else 28  # sección 12

DIA = MINUTES_PER_DAY  # 1440 minutos
NIVELES = range(1, 6)
# Colores de engine/report.py: lo simulado en azul, la teoría o la referencia en naranja,
# una segunda serie en aqua y lo secundario en gris. Los niveles llevan los colores de
# Manchester: rojo, naranja, amarillo, verde y azul.
NIVEL_COLOR = {1: "#d03b3b", 2: "#eb6834", 3: "#e3b505", 4: "#1baf7a", 5: "#2a78d6"}
BANDAS = [f"{lo}–{hi}" for lo, hi in (age_range(PERFILES.population, b) for b in range(4))]


def nombre_dia(d):
    """Nombre corto y fecha del día d de la ejecución (el día 0 es el lunes 5 de enero)."""
    fecha = LLEGADAS.start_date + timedelta(days=int(d))
    return f"{WEEKDAYS[fecha.weekday()][:3]} {fecha.day}"


def ejecuta(centro, perfiles, dias, semilla, cal=CAL):
    """Simula el modelo clínico dias + DRAIN_DAYS días (el día de vaciado de la
    calibración: los llegados al final de la ventana pueden salir) y devuelve la tabla de
    episodios y el ground truth. Los eventos se liberan al salir de la función."""
    sink = MemorySink()
    resultado = simulate_ed(
        centro,
        LLEGADAS,
        dias + DRAIN_DAYS,
        semilla,
        sink=sink,
        warmup_days=cal,
        profiles=perfiles,
    )
    return episode_table(sink.events), resultado.ground_truth


plt.rcParams["figure.figsize"] = (9, 4)
print("Modo rápido" if RAPIDO else "Modo normal")

## 2. Del paciente sin rasgos al paciente con perfil

En el hito 3 un paciente era un nivel y unas duraciones sorteadas de tablas por nivel. En
el hito 4 cada atributo se sortea **condicionado a los anteriores**, en un orden fijo y
con un flujo aleatorio por fuente (decisión 43):

| Paso | Qué se sortea | Depende de | Flujo |
| --- | --- | --- | --- |
| 1 | Contexto: franja, tipo de día, estación | La hora de llegada | — |
| 2 | Banda de edad, edad, sexo, medio de llegada | Contexto | 5 |
| 3 | Motivo de consulta (17) | Banda y contexto | 6 |
| 4 | Gravedad latente G | Motivo y banda | 7 |
| 5 | Nivel asignado = umbrales(G + ε); nivel real = umbrales(G) | G y el error de la enfermera | 1 |
| 6 | Pruebas, resultados y respuestas; duraciones | G, nivel real, motivo y edad | 3 y 2 |

El hospital solo ve una parte: el log lleva la edad, el sexo, el medio, el motivo, el
nivel **asignado** y las pruebas, pero nunca G, el nivel real ni la calidad del triaje,
que van al ground truth (decisión 28). Abajo, un paciente infratriado de tres días de
simulación: su verdad clínica y lo que el log dice de él.

In [ ]:
SEMILLA = 2026
sink = MemorySink()
corta = simulate_ed(CENTRO, LLEGADAS, 3, SEMILLA, sink=sink, warmup_days=1, profiles=PERFILES)
eventos = sink.events
infratriados = [
    e
    for e in corta.ground_truth.episodes
    if e.clinical.triage.quality == UNDER and any(e.clinical.workup.ordered)
]
paciente = infratriados[0]
v = paciente.clinical
print(f"{paciente.episode_id}, {len(corta.ground_truth.episodes)} pacientes en 3 días")
print(f"  contexto  {v.context.label}")
sexo = "mujer" if v.person.female else "hombre"
medio = "ambulancia" if v.person.ambulance else "propios medios"
print(f"  persona   banda {BANDAS[v.person.band]}, {v.person.age} años, {sexo}, {medio}")
print(f"  motivo    {v.reason}")
tri = v.triage
print(
    f"  gravedad  G = {tri.severity:.2f} (z = {tri.z:.2f}); nivel real {tri.true_level},"
    f" asignado {tri.level}: {tri.quality}, salto {tri.jump}"
)
print(f"  pruebas   {[t for t, o in zip(TEST_TYPES, v.workup.ordered, strict=True) if o]}")
print(
    f"  duraciones: triaje {paciente.triage_time:.1f},"
    f" valoración {paciente.assessment_time:.1f} min"
)
print("\nLo que dice el log (los eventos con datos clínicos):")
for e in eventos:
    if e.episode_id == paciente.episode_id and e.event_type in (
        "patient_arrived",
        "patient_registered",
        "triage_completed",
        "test_ordered",
    ):
        print("  " + to_json(e))

In [ ]:
# El nivel asignado depende de la edad a través del motivo y de G: los mayores llegan con
# motivos más graves (disnea, dolor torácico) y más gravedad dentro de cada motivo.
reparto = np.zeros((4, 5))
for e in corta.ground_truth.episodes:
    reparto[e.clinical.person.band, e.level - 1] += 1
reparto = 100 * reparto / reparto.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(9, 3.6))
izquierda = np.zeros(4)
for k in NIVELES:
    ax.barh(BANDAS, reparto[:, k - 1], left=izquierda, color=NIVEL_COLOR[k], label=f"nivel {k}")
    izquierda += reparto[:, k - 1]
ax.set_xlabel("% de los pacientes de la banda")
ax.set_ylabel("banda de edad (años)")
ax.set_title(
    f"Nivel asignado por banda de edad ({len(corta.ground_truth.episodes)} pacientes, 3 días)"
)
ax.legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.25))
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()
for b in range(4):
    print(
        f"{BANDAS[b]:>6}: niveles 1-2 {reparto[b, :2].sum():4.1f} %, nivel 4 {reparto[b, 3]:4.1f} %"
    )

**Para experimentar**

- Elige otro paciente de `infratriados`. ¿Qué motivos aparecen más entre los infratriados?
- Busca un paciente de 80 o más años: ¿qué motivo tiene? ¿Y qué nivel?

> **Lo que debes recordar:** cada atributo se sortea condicionado a los anteriores, en orden fijo y con un flujo por fuente; el log solo lleva lo que ve el hospital, y la gravedad, el nivel real y la calidad del triaje quedan en el ground truth.

## 3. La mezcla de motivos

El peso del motivo m para un paciente de banda b en el contexto c es

> w_m = peso_base_m × f_edad[b] × f_franja × f_día × f_estación

y su probabilidad, w_m / Σ w. En logaritmos los factores se suman: **un factor multiplica,
no suma puntos**. Abajo, los 17 motivos en cuatro situaciones: el mismo modelo da mezclas
muy distintas según quién llega y cuándo.

In [ ]:
SITUACIONES = (
    ("15–44 años, noche de fin de semana de verano", ArrivalContext(2, 1, 2), 0),
    ("15–44 años, mañana laborable de invierno", ArrivalContext(0, 0, 0), 0),
    ("80–99 años, mañana laborable de invierno", ArrivalContext(0, 0, 0), 3),
    ("80–99 años, tarde laborable de verano", ArrivalContext(1, 0, 2), 3),
)
nombres = [r.name for r in PERFILES.reasons]
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(REASON_KEYS))
alto = 0.2
colores = (SECOND_COLOR, OBSERVED_COLOR, THEORY_COLOR, "#7b5ea7")
for i, (etiqueta, contexto, banda) in enumerate(SITUACIONES):
    p = reason_probabilities(PERFILES, contexto, banda)
    ax.barh(y + (i - 1.5) * alto, 100 * p, height=alto, color=colores[i], label=etiqueta)
ax.set_yticks(y, nombres)
ax.invert_yaxis()
ax.set_xlabel("probabilidad del motivo (%)")
ax.set_title("La misma tabla de pesos, cuatro mezclas de motivos")
ax.legend(fontsize=8, loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

**Peso base frente a reparto global.** El reparto del catálogo (la columna «peso aprox.»
del diseño) es lo que sale **en un año**: la media de todos los contextos, ponderada por
las llegadas y por la mezcla de edades de cada uno. Los pesos base son el punto fijo que
hace que salga ese reparto (`base_weights_for`), y no coinciden con él: un motivo de
mayores, como la disnea, necesita menos peso base, porque sus factores de edad lo suben.

In [ ]:
base = np.array([r.base_weight for r in PERFILES.reasons])
base = 100 * base / base.sum()
global_ = 100 * global_reason_shares(PERFILES, LLEGADAS)
catalogo = 100 * catalog_shares()
print(f"{'motivo':<34}{'peso base':>10}{'reparto del año':>17}{'catálogo':>10}")
for k in np.argsort(-catalogo):
    print(f"{nombres[k]:<34}{base[k]:>9.2f}%{global_[k]:>16.2f}%{catalogo[k]:>9.2f}%")
print(
    f"\nMayor diferencia entre el año y el catálogo: {np.abs(global_ - catalogo).max():.4f} puntos"
)

**Para experimentar**

- Cambia la banda de la tercera situación a 1 (45–64 años). ¿Qué motivo cae más?
- ¿Qué motivos tienen un peso base mayor que su reparto del año? ¿Qué tienen en común?

> **Lo que debes recordar:** un factor multiplica el peso y la probabilidad se renormaliza en cada contexto; el reparto del catálogo es la media de un año, y los pesos base se derivan para reproducirlo, no se copian de él.

## 4. Gumbel-max y los números aleatorios comunes: la ola de gripe

Un escenario clínico cambia pesos: una ola de gripe multiplica la disnea por 1,5. Con la
misma semilla, lo ideal es que **solo cambien los pacientes a los que afecta el cambio**,
para que la diferencia entre los dos escenarios sea el efecto y no ruido. Dos formas de
sortear la misma categórica se comportan muy distinto:

- **CDF inversa:** una uniforme u y la primera categoría cuyo acumulado la supera. Si la
  disnea crece, todos los intervalos que vienen detrás se desplazan, y cambian también
  pacientes que nada tienen que ver con la disnea (de traumatismo a dolor abdominal, por
  ejemplo).
- **Gumbel-max** (lo que usa EDsim): un sorteo de Gumbel por categoría y gana el mayor
  log w + g. Si solo crece la disnea, solo cambian los que **pasan a disnea**.

Ningún método puede cambiar menos pacientes que la **distancia de variación total**,
½ Σ |p − q|, y cuando solo crece un peso Gumbel-max la alcanza **exactamente**. El paquete
calcula las tres cifras de forma exacta (`profiles/coupling.py`) y también las simula con
números aleatorios comunes, con el `PersonReasonSampler` del motor.

In [ ]:
SEMILLA = 43
GRIPE = with_base_weights(
    PERFILES, {r.key: r.base_weight * (1.5 if r.key == "disnea" else 1) for r in PERFILES.reasons}
)
exacto = reason_shock_changes(PERFILES, GRIPE, LLEGADAS)
simulado = simulate_reason_shock(PERFILES, GRIPE, LLEGADAS, seed=SEMILLA, patients=PACIENTES_GRIPE)
disnea = REASON_KEYS.index("disnea")
antes = 100 * global_reason_shares(PERFILES, LLEGADAS)[disnea]
despues = 100 * global_reason_shares(GRIPE, LLEGADAS)[disnea]
print(f"Disnea en el año: {antes:.2f} % → {despues:.2f} %")
print(
    f"Cambian de motivo (exacto): mínimo posible {100 * exacto.minimum:.2f} %,"
    f" Gumbel-max {100 * exacto.gumbel:.2f} %, CDF inversa {100 * exacto.inverse_cdf:.2f} %"
)
n = simulado.patients
print(
    f"Simulado con {n} pacientes: Gumbel-max {simulado.gumbel} ({100 * simulado.gumbel / n:.2f} %),"
    f" CDF inversa {simulado.inverse_cdf} ({100 * simulado.inverse_cdf / n:.2f} %)"
)

# Un contexto: mayores de 80 años, un martes de invierno por la mañana.
martes = ArrivalContext(0, 0, 0)
w_antes, w_despues = reason_weights(PERFILES, martes, 3), reason_weights(GRIPE, martes, 3)
contexto = {
    "mínimo": total_variation(w_antes, w_despues),
    "Gumbel-max": gumbel_change_share(w_antes, w_despues),
    "CDF inversa": inverse_cdf_change_share(w_antes, w_despues),
}
sim_contexto = simulate_changes(w_antes, w_despues, PACIENTES_CONTEXTO, SEMILLA)
p_antes = 100 * reason_probabilities(PERFILES, martes, 3)[disnea]
p_despues = 100 * reason_probabilities(GRIPE, martes, 3)[disnea]
print(f"\n80–99 años, martes de invierno por la mañana: disnea {p_antes:.2f} % → {p_despues:.2f} %")
print("  exacto: " + ", ".join(f"{k} {100 * v:.2f} %" for k, v in contexto.items()))

In [ ]:
def barras(ax, titulo, minimo, gumbel, inversa, sim_g, sim_i, n):
    """Lo exacto en barras y lo simulado como puntos con su IC de Wilson al 95 %."""
    exactos = [100 * minimo, 100 * gumbel, 100 * inversa]
    ax.bar([0, 1, 2], exactos, color=[MUTED_COLOR, OBSERVED_COLOR, THEORY_COLOR], width=0.6)
    for x, k in ((1, sim_g), (2, sim_i)):
        lo, hi = share_interval(k, n)
        ax.errorbar(
            x,
            100 * k / n,
            yerr=[[100 * (k / n - lo)], [100 * (hi - k / n)]],
            fmt="o",
            color="k",
            capsize=4,
            label="simulado (IC 95 %)" if x == 1 else None,
        )
    for x, v in enumerate(exactos):
        ax.text(x, v + 0.4, f"{v:.2f} %", ha="center", fontsize=9)
    ax.set_xticks([0, 1, 2], ["mínimo posible", "Gumbel-max", "CDF inversa"])
    ax.set_ylabel("pacientes que cambian de motivo (%)")
    ax.set_title(titulo, fontsize=10)
    ax.grid(axis="y", alpha=0.3)


fig, (ax_a, ax_c) = plt.subplots(1, 2, figsize=(12, 4.2))
barras(
    ax_a,
    f"Un año ({simulado.patients:,} pacientes simulados)".replace(",", "."),
    exacto.minimum,
    exacto.gumbel,
    exacto.inverse_cdf,
    simulado.gumbel,
    simulado.inverse_cdf,
    simulado.patients,
)
barras(
    ax_c,
    f"80–99 años, martes de invierno ({sim_contexto.patients:,} simulados)".replace(",", "."),
    contexto["mínimo"],
    contexto["Gumbel-max"],
    contexto["CDF inversa"],
    sim_contexto.gumbel,
    sim_contexto.inverse_cdf,
    sim_contexto.patients,
)
ax_a.legend(fontsize=8)
fig.suptitle("Disnea ×1,5: lo exacto (barras) y lo simulado con números aleatorios comunes")
plt.tight_layout()
plt.show()

En la ejecución normal (un año, 109.542 pacientes simulados), la disnea pasa del 8,06 %
al 11,43 % de los pacientes. Gumbel-max cambia exactamente el mínimo posible, el 3,37 %
(simulado: 3,41 %), y la CDF inversa, el 15,55 % (simulado: 15,52 %): más de cuatro veces
más pacientes, casi todos sin nada que ver con la disnea. En los mayores de 80 años un
martes de invierno por la mañana, donde la disnea pesa más (del 19,95 al 27,21 %), la
distancia crece: 7,26 % frente a 30,54 %. La CDF inversa depende además del orden de las
categorías (aquí, el de `REASON_KEYS`); Gumbel-max no tiene orden.

**Para experimentar**

- Cambia el factor de la gripe a 3. ¿Sigue coincidiendo Gumbel-max con el mínimo? ¿Por qué?
- Cambia **dos** pesos a la vez (disnea ×1,5 y traumatismo ×0,8). ¿Sigue Gumbel-max en el
  mínimo? (Pista: ahora unos entran en disnea y otros salen del traumatismo.)

> **Lo que debes recordar:** Gumbel-max y la CDF inversa sortean la misma categórica, pero ante un cambio de pesos la CDF inversa mueve a pacientes que no tienen nada que ver; con Gumbel-max y un solo peso que crece solo cambian los que pasan a esa categoría, que es el mínimo posible, y por eso los escenarios clínicos se comparan con números aleatorios comunes.

## 5. Gravedad y triaje

Cada paciente tiene una gravedad G que nadie ve; la enfermera clasifica G + ε, con
ε = sesgo + ruido · z. Los cinco niveles son cuatro umbrales sobre esa escala: nivel 1 por
encima de c₁, nivel 5 por debajo de c₄. El **nivel asignado** corta G + ε y el **real**,
G. Los umbrales se derivan para que el reparto **asignado** de un año sea el objetivo
(Emergencias 2013: 0,4 / 12,1 / 33,6 / 52,6 / 1,4 %), y se escriben en el TOML: no se
recalculan al cargar, para que un escenario más grave dé más niveles urgentes.

Abajo, la densidad de G en un año (una mezcla de 68 normales, una por motivo y banda) y
la de G + ε, más ancha por el error, con los umbrales escritos.

In [ ]:
t = PERFILES.triage
mezcla = severity_mixture(PERFILES, LLEGADAS)
observada = SeverityMixture(mezcla.weights, mezcla.means + t.bias, np.hypot(mezcla.sds, t.noise))
x = np.linspace(-3.5, 5, 600)
matriz = triage_matrix(mezcla, t.thresholds, t.noise, t.bias)
condicional = 100 * matriz / matriz.sum(axis=1, keepdims=True)  # P(asignado | real)

fig, (ax_g, ax_m) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1.4, 1]})
ax_g.plot(x, mezcla.pdf(x), color=OBSERVED_COLOR, lw=2, label="G (real)")
ax_g.plot(
    x,
    observada.pdf(x),
    color=THEORY_COLOR,
    lw=1.5,
    ls="--",
    label="G + ε (lo que clasifica la enfermera)",
)
ax_g.set_ylim(0, 0.5)
for k, c in enumerate(t.thresholds, start=1):
    ax_g.axvline(c, color="k", lw=0.8, ls=":")
    ax_g.text(c, 0.47, f"c{k}", ha="center", fontsize=9, backgroundcolor="white")
ax_g.set_xlabel("gravedad G (desviaciones de la población de referencia)")
ax_g.set_ylabel("densidad")
ax_g.set_title("Gravedad de un año y umbrales del triaje")
ax_g.legend(fontsize=8)
im = ax_m.imshow(condicional, cmap="Blues", vmin=0, vmax=100)
for i in range(5):
    for j in range(5):
        ax_m.text(
            j,
            i,
            f"{condicional[i, j]:.1f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if condicional[i, j] > 60 else "k",
        )
ax_m.set_xticks(range(5), [str(k) for k in NIVELES])
ax_m.set_yticks(range(5), [str(k) for k in NIVELES])
ax_m.set_xlabel("nivel asignado")
ax_m.set_ylabel("nivel real")
ax_m.set_title("P(asignado | real), % por fila")
fig.colorbar(im, ax=ax_m, label="%")
plt.tight_layout()
plt.show()
infra, sobre = triage_error_rates(matriz)
print(f"Umbrales {list(t.thresholds)}; ruido {t.noise}, sesgo {t.bias}")
print(f"Infratriaje {100 * infra:.2f} %, sobretriaje {100 * sobre:.2f} %")
print(f"Reparto asignado {np.round(100 * matriz.sum(axis=0), 2)} %")
print(f"Reparto real     {np.round(100 * matriz.sum(axis=1), 2)} %")

**Mover el ruido y el sesgo con los umbrales escritos.** El ruido y el sesgo están
calibrados contra la concordancia del Manchester en Asturias (10,1 % de infratriaje y
6,3 % de sobretriaje). Si una enfermera más insegura tuviera más ruido, con los mismos
umbrales cambiarían las dos tasas y también el reparto asignado: el ruido «inventa»
niveles 1 y 5, porque ensancha G + ε.

In [ ]:
ruidos = np.linspace(0.05, 0.8, 31)
sesgos = np.linspace(-0.4, 0.4, 33)
por_ruido = np.array(
    [
        [
            *triage_error_rates(triage_matrix(mezcla, t.thresholds, r, t.bias)),
            *assigned_shares(mezcla, t.thresholds, r, t.bias)[[0, 4]],
        ]
        for r in ruidos
    ]
)
por_sesgo = np.array(
    [triage_error_rates(triage_matrix(mezcla, t.thresholds, t.noise, s)) for s in sesgos]
)
fig, (ax_r, ax_s) = plt.subplots(1, 2, figsize=(12, 4))
ax_r.plot(ruidos, 100 * por_ruido[:, 0], color=THEORY_COLOR, lw=2, label="infratriaje")
ax_r.plot(ruidos, 100 * por_ruido[:, 1], color=OBSERVED_COLOR, lw=2, label="sobretriaje")
ax_r.plot(
    ruidos, 100 * por_ruido[:, 3], color=NIVEL_COLOR[5], lw=1, ls="--", label="nivel 5 asignado"
)
ax_r.axvline(t.noise, color="k", lw=0.8, ls=":")
ax_r.set_xlabel("ruido del triaje (unidades de G)")
ax_r.set_ylabel("% de los pacientes")
ax_r.set_title(f"Más ruido, con el sesgo {t.bias} y los umbrales escritos")
ax_r.legend(fontsize=8)
ax_s.plot(sesgos, 100 * por_sesgo[:, 0], color=THEORY_COLOR, lw=2, label="infratriaje")
ax_s.plot(sesgos, 100 * por_sesgo[:, 1], color=OBSERVED_COLOR, lw=2, label="sobretriaje")
ax_s.axvline(t.bias, color="k", lw=0.8, ls=":")
ax_s.set_xlabel("sesgo del triaje (negativo: tiende a infratriar)")
ax_s.set_ylabel("% de los pacientes")
ax_s.set_title(f"Otro sesgo, con el ruido {t.noise} y los umbrales escritos")
ax_s.legend(fontsize=8)
for ax in (ax_r, ax_s):
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
i = int(np.argmin(np.abs(ruidos - 2 * t.noise)))
print(
    f"Con el doble de ruido ({ruidos[i]:.2f}): infratriaje {100 * por_ruido[i, 0]:.1f} %,"
    f" sobretriaje {100 * por_ruido[i, 1]:.1f} %, nivel 5 asignado {100 * por_ruido[i, 3]:.2f} %"
)

Con los valores escritos, el infratriaje es del 10,10 % y el sobretriaje del 6,29 %. La
densidad de G + ε es más ancha que la de G y está un poco desplazada a la izquierda por el
sesgo negativo. La matriz muestra dónde se concentran los errores: el 9,2 % de los niveles 1
reales se asigna como 2 y el 19,5 % de los niveles 5 reales sube a 4, porque los niveles
raros ocupan franjas estrechas de G y el ruido las cruza con facilidad. El reparto real
(0,39 / 12,51 / 35,82 / 50,62 / 0,65 %) es más estrecho que el asignado (0,40 / 12,10 /
33,56 / 52,54 / 1,40 %): el ruido «inventa» niveles 1 y 5. Con el doble de ruido (0,68),
el infratriaje sube al 16,8 %, el sobretriaje al 15,0 % y el nivel 5 asignado, del 1,4 % al
3,4 %.

**Para experimentar**

- ¿Con qué sesgo saldrían iguales el infratriaje y el sobretriaje? ¿Por qué un ruido
  simétrico sin sesgo da más sobretriaje que infratriaje?
- Mira la fila del nivel real 5 de la matriz: ¿por qué un 5 real sube a 4 tan a menudo?

> **Lo que debes recordar:** el nivel asignado es G + ε cortado por umbrales y el real, G con los mismos umbrales; los errores se concentran en los niveles raros, de franjas estrechas, y con los umbrales escritos más ruido cambia el infratriaje, el sobretriaje y el propio reparto asignado.

## 6. El nivel 1 en la puerta y los críticos no reconocidos

Desde la tarea 4.1 (decisión 39), un paciente con **nivel 1 asignado** se clasifica en la
puerta al llegar, sin enfermera, y va directo al box. Pero la puerta decide con el nivel
asignado, no con el real: un nivel 1 real al que la enfermera asigna un 2 no pasa por la
puerta. Hace cola de triaje como cualquier otro y después espera box y médico como un 2.
Es el crítico no reconocido.

Lo exacto, de la matriz de la sección 5: 9,2 de cada 100 niveles 1 reales se
asignan como 2. En el centro calibrado son pocos pacientes, así que la ejecución principal
dura 91 días. Esta ejecución se usa también en las secciones 8, 10 y 11.

In [ ]:
SEMILLA_PRINCIPAL = 2026
tabla, verdad = ejecuta(CENTRO, PERFILES, DIAS, SEMILLA_PRINCIPAL)
print(
    f"{DIAS} días ({CAL} de calentamiento y {DRAIN_DAYS} de vaciado), semilla {SEMILLA_PRINCIPAL}:"
    f" {len(tabla)} pacientes"
)
puerta = true_level_waits(tabla, verdad, "door_to_doctor", CAL, DIAS)
pretriaje = true_level_waits(tabla, verdad, "triage_wait", CAL, DIAS)
reales_1 = puerta.true == 1
grupos = {
    "real 1, asignado 1 (puerta)": reales_1 & (puerta.assigned == 1),
    "real 1, asignado 2 (no reconocido)": reales_1 & (puerta.assigned == 2),
}
print(f"Niveles 1 reales en la ventana: {reales_1.sum()}")
for nombre, g in grupos.items():
    if g.sum() == 0:
        print(f"  {nombre}: ninguno" + (" en modo rápido" if RAPIDO else ""))
        continue
    a = puerta.values[g]
    print(
        f"  {nombre}: {g.sum()} pacientes; pretriaje medio {pretriaje.values[g].mean():.1f} min,"
        f" llegada-médico (reloj A) medio {a.mean():.1f} min, máximo {a.max():.1f}"
    )
print(f"Exacto: el {condicional[0, 1]:.1f} % de los niveles 1 reales se asigna como 2")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, g in enumerate(grupos.values()):
    valores = puerta.values[g]
    jitter = np.random.default_rng(i).uniform(-0.15, 0.15, valores.size)
    ax.scatter(valores, i + jitter, s=18, color=NIVEL_COLOR[i + 1], alpha=0.8)
    if valores.size:
        ax.plot([valores.mean()] * 2, [i - 0.3, i + 0.3], color="k", lw=2)
ax.set_yticks([0, 1], list(grupos))
ax.set_xlabel("de la llegada al primer médico, reloj A (min)")
ax.set_title(f"Los niveles 1 reales, según los vio la enfermera (raya: media; {DIAS - CAL} días)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

En la ejecución normal (89 días medidos) hay 141 niveles 1 reales. Los 126 que la
enfermera reconoce van a la puerta y ven al médico en 1,7 minutos de media: lo que les
queda es la espera de médico, porque los médicos no se desalojan (decisión 25). Los 15
asignados como 2 (un 10,6 %, cerca del 9,2 % exacto) esperan 9,1 minutos de pretriaje de
media y ven al médico a los 19,8 minutos; uno, a los 92. Es el coste de un crítico no
reconocido, y solo se ve con el ground truth: en el log son niveles 2.

**Para experimentar**

- ¿Qué motivo tienen los niveles 1 reales asignados como 2? Búscalos en `verdad` con su
  `episode_id`.
- Si la enfermera tuviera menos ruido, ¿cuántos niveles 1 reales irían a la puerta?
  (Pista: `triage_matrix` con otro ruido, sección 5.)

> **Lo que debes recordar:** la puerta decide con el nivel asignado; un crítico infratriado como 2 hace la cola del triaje y espera como un 2, y ese coste solo se ve con el ground truth.

## 7. Pruebas y duraciones

**Pruebas.** Cada tipo (analítica, radiografía, ECG e imagen) se pide con una probabilidad
logística sobre la gravedad real, el motivo y la edad:
P = 1 / (1 + e^−(a + b·G + c·mayor)). El TOML da el porcentaje para un adulto de menos de
65 años con G = 0 de cada motivo, y dos odds ratios por tipo: por unidad de G y por ser
mayor. La logística convierte una suma sin límites en una probabilidad entre 0 y 1.

**Duraciones.** La valoración y la reevaluación son lognormales con una mediana y un p90
por **nivel real**, y cada motivo multiplica la mediana por un factor k. El médico atiende
al paciente que tiene delante, no al nivel que le asignaron.

In [ ]:
MOTIVO = "dolor_toracico"
NOMBRE_PRUEBA = {
    "analitica": "Analítica",
    "radiografia": "Radiografía",
    "ecg": "ECG",
    "imagen": "Imagen (eco o TC)",
}
motivo = PERFILES.reasons[REASON_KEYS.index(MOTIVO)]
g = np.linspace(-2.5, 3.5, 121)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, prueba, pct in zip(axes, PERFILES.diagnostic_tests, motivo.test_pct, strict=True):
    for mayor, color, etiqueta in (
        (False, OBSERVED_COLOR, "15–64 años"),
        (True, THEORY_COLOR, "65 o más"),
    ):
        ax.plot(
            g,
            [100 * order_probability(prueba, pct, x, mayor) for x in g],
            color=color,
            lw=2,
            label=etiqueta,
        )
    ax.axvline(0, color=MUTED_COLOR, lw=0.8, ls=":")
    ax.set_title(
        f"{NOMBRE_PRUEBA[prueba.key]} (OR por G {prueba.severity_or:g}, mayor {prueba.older_or:g})",
        fontsize=9,
    )
    ax.set_xlabel("gravedad real G")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("probabilidad de pedirla (%)")
axes[0].legend(fontsize=8)
fig.suptitle(f"Pruebas de un paciente con {motivo.name.lower()}, según su gravedad y su edad")
plt.tight_layout()
plt.show()
modelo_pruebas = workup_model(PERFILES, LLEGADAS)
por_nivel = 100 * modelo_pruebas.order_by_true_level()
print("% al que se pide cada prueba por nivel real (exacto, un año):")
print(f"{'nivel':>6}" + "".join(f"{NOMBRE_PRUEBA[t].split(' (')[0]:>13}" for t in TEST_TYPES))
for k in NIVELES:
    print(f"{k:>6}" + "".join(f"{por_nivel[k - 1, j]:>12.1f}%" for j in range(len(TEST_TYPES))))

In [ ]:
minutos = np.linspace(0.1, 150, 600)
modelo_duraciones = duration_model(PERFILES, LLEGADAS)
fig, (ax_v, ax_k) = plt.subplots(1, 2, figsize=(13, 4))
for k in NIVELES:
    mediana, p90 = PERFILES.durations.assessment[k - 1]
    ax_v.plot(
        minutos,
        lognormal_pdf(minutos, *duration_params(mediana, p90)),
        color=NIVEL_COLOR[k],
        lw=2,
        label=f"nivel real {k}: mediana {mediana:g}, p90 {p90:g}",
    )
ax_v.set_xlabel("valoración (min)")
ax_v.set_ylabel("densidad")
ax_v.set_title("Valoración por nivel real")
ax_v.legend(fontsize=8)
mediana, p90 = PERFILES.durations.assessment[2]  # nivel real 3
for clave, color in (
    ("orl", SECOND_COLOR),
    ("dolor_toracico", MUTED_COLOR),
    ("psiquiatrico", THEORY_COLOR),
):
    r = PERFILES.reasons[REASON_KEYS.index(clave)]
    ax_k.plot(
        minutos,
        lognormal_pdf(minutos, *duration_params(mediana, p90, r.duration_factor)),
        color=color,
        lw=2,
        label=f"{r.name}: k = {r.duration_factor:g}",
    )
ax_k.set_xlabel("valoración (min)")
ax_k.set_ylabel("densidad")
ax_k.set_title("Nivel real 3, tres motivos: k multiplica la mediana, σ no cambia")
ax_k.legend(fontsize=8)
for ax in (ax_v, ax_k):
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
medias = modelo_duraciones.mean_by_level("valoracion")
print(
    "Valoración media por nivel real (con los k): "
    + " / ".join(f"{m:.1f}" for m in medias)
    + " min"
)
print(f"Carga del médico por paciente: {modelo_duraciones.physician_load().load:.2f} min")

**Para experimentar**

- Cambia `MOTIVO` a `"traumatismo"`: ¿qué prueba domina? ¿Cambia algo con la edad?
- Con un OR por G de 3,3, ¿cuánto sube la probabilidad de pedir analítica al pasar de
  G = 0 a G = 1 en un adulto?

> **Lo que debes recordar:** las pruebas y las duraciones dependen de la gravedad real y del motivo, no del nivel asignado; un infratriado espera como su nivel asignado pero consume como su nivel real, y ese es el mecanismo que hace peligroso el infratriaje.

## 8. Los tres relojes

Entre la llegada y el primer médico hay tres tramos: el pretriaje, el triaje y la espera de
box y médico. En España se usan tres relojes distintos (decisión 40):

| Reloj | Medida | Desde | Para qué |
| --- | --- | --- | --- |
| A | `door_to_doctor` | La llegada | El TPAM del SAS |
| B | `triage_start_to_doctor` | El inicio del triaje | El TEA del SAS |
| C | `classified_to_doctor` | La clasificación | El cumplimiento Manchester de EDsim |

El cumplimiento Manchester se mide con C: antes del triaje el nivel no existe, y no se
puede incumplir un plazo que aún no se ha asignado. Además, C separa responsabilidades:
el pretriaje es de la enfermería de triaje y C, de boxes y médicos. Abajo, el mismo día
medido con los tres relojes.

In [ ]:
DIA_RELOJES = CAL + 1 if RAPIDO else 7  # el lunes 12 de enero en modo normal
cumplimiento = {c: manchester_compliance(tabla, CAL, DIAS, measure=c) for c in CLOCKS}
fila = DIA_RELOJES - CAL
etiquetas = {
    "door_to_doctor": "A, desde la llegada",
    "triage_start_to_doctor": "B, desde el inicio del triaje",
    "classified_to_doctor": "C, desde la clasificación",
}
fig, ax = plt.subplots(figsize=(10, 4))
ancho = 0.26
for i, (c, color) in enumerate(
    zip(CLOCKS, (THEORY_COLOR, SECOND_COLOR, OBSERVED_COLOR), strict=True)
):
    within = cumplimiento[c].within[fila, 1:]
    ax.bar(
        np.arange(2, 6) + (i - 1) * ancho,
        100 * within,
        width=ancho,
        color=color,
        label=etiquetas[c],
    )
ax.axhline(90, color="k", lw=1, ls="--", label="objetivo: 90 % de los pacientes")
ax.set_xticks(
    range(2, 6), [f"nivel {k}\n(máx. {MANCHESTER_MAX_WAIT[k - 1]:g} min)" for k in range(2, 6)]
)
ax.set_ylabel("% dentro del máximo de Manchester")
ax.set_ylim(0, 105)
ax.set_title(f"{nombre_dia(DIA_RELOJES)} de enero: el mismo día con los tres relojes")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()
tarde = classified_late(tabla, CAL, DIAS)
print(f"{nombre_dia(DIA_RELOJES)}: % dentro del máximo, niveles 2 a 5")
for c in CLOCKS:
    print(
        f"  {etiquetas[c]:<32}"
        + "  ".join(f"{100 * x:5.1f}" for x in cumplimiento[c].within[fila, 1:])
    )
print(f"\nEn los {DIAS - CAL} días, días fuera de Manchester por nivel (2 a 5):")
for c in CLOCKS:
    out = cumplimiento[c].out.sum(axis=0)[1:]
    print(f"  {etiquetas[c]:<32}" + "  ".join(f"{x:4d}" for x in out))
print(
    "\nNiveles 2 ya fuera de sus 10 min al clasificarse (pretriaje y triaje):"
    f" {100 * tarde[1]:.1f} %;"
    f" niveles 3 fuera de sus 60: {100 * tarde[2]:.2f} %"
)
nivel_1 = clock_quantiles(tabla, CAL, DIAS)[0]
print(
    f"Nivel 1, sin porcentaje (su máximo es 0): {nivel_1.n} pacientes; reloj A mediana"
    f" {nivel_1.median['door_to_doctor']:.1f} y p90 {nivel_1.p90['door_to_doctor']:.1f} min"
)

En la ejecución normal, el lunes 12 de enero el 92,2 % de los niveles 2 ve al médico en
10 minutos desde la clasificación (C), pero solo el 52,9 % desde el inicio del triaje (B) y
el 11,8 % desde la llegada (A). En los 89 días, el nivel 2 está fuera de Manchester con el
reloj A **todos los días** (89 de 89) y con el C, 34. La razón: el 45,5 % de los niveles 2
ya pasa de sus 10 minutos **al clasificarse**, entre el pretriaje y el triaje (mediana de
5); ningún médico puede hacerlos cumplir con el reloj A. En los niveles 4 y 5 los tres relojes casi coinciden: su espera es de box y
médico, y el pretriaje pesa poco frente a 120 o 240 minutos. El nivel 1 se informa solo
con su tiempo (con el reloj A, mediana de 0,7 minutos y p90 de 6,0), porque su máximo es 0
y un porcentaje no diría nada.

**Para experimentar**

- Elige un día de fin de semana (`DIA_RELOJES = 5`, el sábado 10). ¿Cambia la distancia
  entre A y C?
- ¿Qué haría falta para que el nivel 2 cumpliera con el reloj A? Prueba más enfermeras
  de triaje en la sección 12.

> **Lo que debes recordar:** el cumplimiento Manchester se mide desde la clasificación (reloj C), porque antes no hay nivel; con el reloj A el nivel 2 no puede cumplir, porque el pretriaje y el triaje ya se comen sus 10 minutos, y eso es un problema de la enfermería de triaje, no de los médicos.

## 9. La calibración

El centro se calibra a **lo observado** en hospitales españoles, no a la norma: un centro
calibrado a la norma cumpliría por diseño y no serviría para el laboratorio. Las cifras
del SAS (CMBD 2013) incluyen menores y son de 2013, así que los objetivos son
**horquillas** anchas (`config/objetivos.toml`), y un objetivo pasa si la media entre
réplicas cae dentro y su IC al 95 % no es más ancho que la horquilla (regla 4).

Hay **dos mandos con objetivos separados** (decisión 41): la enfermería de triaje mueve el
pretriaje y casi nada más; los médicos mueven la espera desde la clasificación y no tocan el
pretriaje. Cada mando sale de un solo número u, una ocupación objetivo del turno: personal
del turno = ⌈carga del turno / u⌉. Primero se ajusta la enfermería y, con ella fija, los
médicos.

**La ventana anual** (tarea 4.6c). Los objetivos del SAS son medias anuales, así que cada
candidato se mide con 12 réplicas, **una por mes**: la m empieza el primer lunes del mes
m+1 y simula 14 días, con 2 de calentamiento. La media entre réplicas es la del año, y el IC
incluye la variación entre meses. La plantilla es la misma todo el año.

**La semilla de comprobación puede vetar, no elegir** (decisión 41, revisada en la 4.6c).
La regla elige con la semilla 2026; si la elegida falla un objetivo de ajuste con la 2027,
se descarta y se toma la siguiente de la regla que pase con las dos.

**Esta calibración no sustituye al registro.** Aquí se evalúan solo las plantillas del
centro y sus vecinas en la regla, con la misma ventana y las mismas semillas que el
registro (`docs/calibracion/registro.md`), donde están las tablas completas, los comandos y
la confirmación con la semilla 2028.

In [ ]:
def evalua(mando, plantillas, semilla):
    """Las plantillas del mando, con el otro como en el centro y la ventana anual."""
    return evaluate_staffings(
        CENTRO,
        LLEGADAS,
        PERFILES,
        OBJETIVOS,
        mando,
        plantillas,
        replicas=REPLICAS_CAL,
        days=DIAS_CAL,
        warmup_days=CAL_CAL,
        seed=semilla,
        start_dates=FECHAS_CAL,
    )


def alrededor(plantillas, actual, antes, despues):
    """La plantilla actual y sus vecinas en la regla (las de antes tienen más personal)."""
    i = [s.staff for s in plantillas].index(actual)
    return plantillas[max(i - antes, 0) : i + despues + 1]


def comprobacion(mando, apuntes):
    """La misma plantilla con la semilla de comprobación, que puede vetar (no elegir);
    apunta cada fila que se pide."""

    def check(fila):
        (comprobada,) = evalua(mando, [fila.staffing], CHECK_SEED)
        apuntes.append(comprobada)
        return comprobada

    return check


def linea(f, objetivo):
    r = f.result(objetivo)
    return (
        f"  {str(list(f.staffing.staff)):<14} u {f.staffing.u_low:.4f}–{f.staffing.u_high:.4f}"
        f"  media {r.mean:7.2f} [{r.summary.ci_low:7.2f}; {r.summary.ci_high:7.2f}]"
        f"  {'pasa' if r.passed else 'no pasa' + (' (IC)' if r.within else '')}"
    )


# Enfermería: la del centro y sus vecinas (una enfermera más por la tarde, una menos por la
# mañana); médicos: los del centro y sus vecinos (un médico más y uno menos).
cargas_enf = knob_loads(NURSES, PERFILES, LLEGADAS, CENTRO.shift_starts)
plantillas_enf = alrededor(rule_staffings(cargas_enf), CENTRO.triage_nurses, 0 if RAPIDO else 1, 1)
filas_enf = evalua(NURSES, plantillas_enf, FIT_SEED)
cargas_med = knob_loads(DOCTORS, PERFILES, LLEGADAS, CENTRO.shift_starts)
plantillas_med = alrededor(rule_staffings(cargas_med), CENTRO.doctors, 1, 0 if RAPIDO else 1)
filas_med = evalua(DOCTORS, plantillas_med, FIT_SEED)

print(f"Cargas por turno: enfermería {np.round(cargas_enf, 3)}, médicos {np.round(cargas_med, 3)}")
print(f"{REPLICAS_CAL} réplicas de {DIAS_CAL} días, desde {[f'{d:%d/%m}' for d in FECHAS_CAL]}")
eleccion = {}
for nombre, mando, filas, objetivo in (
    ("Enfermería", NURSES, filas_enf, "pretriaje_medio"),
    ("Médicos", DOCTORS, filas_med, "tiempo_medico_niveles_2_5"),
):
    print(f"\n{nombre}, {objetivo}, semilla de ajuste {FIT_SEED}:")
    for f in filas:
        print(linea(f, objetivo))
    sin_veto = choose_staffing(filas, mando)
    pedidas = []
    eleccion[mando] = choose_staffing(filas, mando, check=comprobacion(mando, pedidas))
    print(f"Con {FIT_SEED} la regla elige", sin_veto and list(sin_veto.staffing.staff))
    print(f"Comprobación con {CHECK_SEED}, en el orden de la regla hasta que una pasa:")
    for f in pedidas:
        print(linea(f, objetivo))
    print("Elección:", None if eleccion[mando] is None else list(eleccion[mando].staffing.staff))
eleccion_enf, eleccion_med = eleccion[NURSES], eleccion[DOCTORS]
print(f"\nEl centro: {list(CENTRO.triage_nurses)} y {list(CENTRO.doctors)}")

In [ ]:
def sensibilidad(ax, filas, nombre, titulo, eje):
    objetivo = next(o for o in OBJETIVOS if o.name == nombre)
    lo, hi = objetivo.band
    ax.axhspan(lo, hi, color=SECOND_COLOR, alpha=0.15, label=f"horquilla {lo:g}–{hi:g}")
    for x, f in enumerate(filas):
        r = f.result(nombre)
        color = OBSERVED_COLOR if r.passed else MUTED_COLOR
        ax.errorbar(
            x,
            r.mean,
            yerr=[[r.mean - r.summary.ci_low], [r.summary.ci_high - r.mean]],
            fmt="o",
            color=color,
            capsize=4,
        )
    actual = [
        f.center.triage_nurses if nombre == "pretriaje_medio" else f.center.doctors for f in filas
    ]
    ax.set_xticks(range(len(filas)), [str(list(s)) for s in actual], fontsize=8)
    ax.set_ylabel(eje)
    ax.set_title(titulo, fontsize=10)
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=0.3)


fig, (ax_e, ax_d) = plt.subplots(1, 2, figsize=(13, 4.2))
sensibilidad(
    ax_e,
    filas_enf,
    "pretriaje_medio",
    "Enfermería de triaje: pretriaje medio",
    "pretriaje medio, niveles 2 a 5 (min)",
)
sensibilidad(
    ax_d,
    filas_med,
    "tiempo_medico_niveles_2_5",
    f"Médicos (enfermería {list(CENTRO.triage_nurses)}): reloj A",
    "llegada-médico medio, niveles 2 a 5 (min)",
)
ax_e.set_xlabel("enfermeras de triaje por turno (mañana, tarde, noche)")
ax_d.set_xlabel("médicos por turno (mañana, tarde, noche)")
fig.suptitle(
    f"Media e IC al 95 % con {REPLICAS_CAL} réplicas (azul: pasa la regla 4; gris: no pasa)"
)
plt.tight_layout()
plt.show()

# El perfil por mes: cada réplica es la quincena de un mes, con la misma plantilla todo el año.
MESES = ("ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic")
fig, ax = plt.subplots(figsize=(11, 3.8))
ancho = 0.8 / len(filas_med)
for j, f in enumerate(filas_med):
    del_centro = f.center.doctors == CENTRO.doctors
    ax.bar(
        np.arange(REPLICAS_CAL) + (j - (len(filas_med) - 1) / 2) * ancho,
        [r.targets["tiempo_medico_niveles_2_5"].value for r in f.replicas],
        width=ancho,
        color=OBSERVED_COLOR if del_centro else (MUTED_COLOR, THEORY_COLOR)[j % 2],
        label=f"médicos {list(f.center.doctors)}" + (" (el centro)" if del_centro else ""),
    )
ax.axhspan(35, 70, color=SECOND_COLOR, alpha=0.15, label="horquilla 35–70 (del año)", zorder=0)
ax.set_xticks(range(REPLICAS_CAL), [f"{MESES[d.month - 1]} {d.day}" for d in FECHAS_CAL])
ax.set_ylabel("llegada-médico medio, niveles 2 a 5 (min)")
ax.set_title(f"Reloj A por mes: la quincena desde el primer lunes (semilla {FIT_SEED})")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
fila_centro = next(f for f in filas_med if f.center.doctors == CENTRO.doctors)
print("Censurados por réplica con el centro:", [r.censored for r in fila_centro.replicas])

**El nivel 3, de ajuste a comprobación.** El SAS da el tiempo medio hasta el médico (TPAM,
reloj A) por nivel. Pero ese TPAM **incluye menores y va por nivel asignado**, así que la
comparación con el modelo, de adultos, es solo de **orden de magnitud**. Lo que importa es
la **forma**: en el SAS los niveles 3 a 5 esperan parecido; en el modelo, el 3 casi no
espera y el 5 espera mucho más.

In [ ]:
reloj_a = true_level_waits(tabla, verdad, "door_to_doctor", CAL, DIAS)
modelo_a = [
    reloj_a.values[reloj_a.assigned == k].mean() if (reloj_a.assigned == k).any() else np.nan
    for k in NIVELES
]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(
    np.arange(1, 6) - 0.2,
    sas.TPAM_BY_LEVEL,
    width=0.4,
    color=THEORY_COLOR,
    label="SAS 2013, TPAM (con menores)",
)
ax.bar(
    np.arange(1, 6) + 0.2,
    modelo_a,
    width=0.4,
    color=OBSERVED_COLOR,
    label=f"modelo, centro calibrado ({DIAS - CAL} días)",
)
ax.set_xticks(range(1, 6), [f"nivel {k}" for k in NIVELES])
ax.set_ylabel("llegada-médico medio, reloj A (min)")
ax.set_title("Tiempo hasta el médico por nivel asignado: la forma, no la cifra")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print("Nivel     SAS   modelo")
for k in NIVELES:
    print(f"{k:>5}  {sas.TPAM_BY_LEVEL[k - 1]:6.1f}  {modelo_a[k - 1]:6.1f}")

En la ejecución normal, con la ventana anual y la semilla 2026, en la enfermería pasan
[3, 2, 1] (pretriaje de 6,78 minutos) y [2, 2, 1] (13,20); con una enfermera más por la
tarde ([3, 3, 1]) el pretriaje baja a 3,5, fuera. La regla elige [2, 2, 1], la más cercana a
la referencia del SAS (12,4), pero con la semilla 2027 su pretriaje sale de la horquilla
(15,16 [10,61; 19,71]): **el veto la descarta** y queda [3, 2, 1], que pasa con las dos
(7,86 con 2027). Con dos enfermeras de mañana el pretriaje depende de unos pocos días de
muchas llegadas, y por eso cambia tanto de una semilla a otra. En los médicos solo pasa
[12, 11, 4] (42,10 minutos): con un médico más de noche ([12, 11, 5]) el año da 27,75, por
debajo de 35, y con uno menos de mañana ([11, 11, 4]) la media (50,23) cae dentro, pero el
IC (de 27,8 a 72,7) es más ancho que la horquilla, porque los meses se separan demasiado.
Las filas de médicos son las del registro, con la misma ventana y las mismas semillas; las
de enfermería se evalúan con los médicos de ahora y solo difieren de las del registro por
los censurados, que no se miden (13,20 frente a 13,21 en [2, 2, 1]).

**Enero, cerca de la saturación.** La gráfica por mes muestra el precio de una plantilla
para todo el año: con [12, 11, 4] la quincena de enero da 105 minutos de reloj A y la de
agosto, 25. De noche, 4 médicos para una carga de 3,4 son un u de 0,85 en el día medio y de
0,93 con el factor de enero (1,10), y 8 pacientes de enero siguen dentro al final del día
de vaciado. Un refuerzo de invierno queda para versiones posteriores. La ejecución principal
de este notebook, de enero a abril, cae justo en los meses de más carga.

Por niveles, el modelo y el SAS tienen formas distintas. En el SAS los niveles 3, 4 y 5
esperan parecido (59,0 / 59,8 / 68,5 minutos); en el modelo, el 3 espera poco (29,2) y el 5
muchísimo más: 627 minutos de media en los 89 días, de los que dos tercios vienen de una
sola semana, la del 19 al 25 de enero (3897 minutos de media en esa semana; sin ella, 224).
Esas 65 horas no son realistas: el modelo aún no tiene abandono sin ser visto
(`patient_left_wbs`, hito 5), y en un hospital real buena parte de esos pacientes se iría.
Lo acotan el abandono del hito 5 y el refuerzo de invierno.
Con **prioridad estricta en un solo grupo de médicos**, el nivel 3 pasa por delante de todos
los 4 y 5, que cargan con las colas; en los hospitales reales los niveles 4 y 5 suelen tener
circuito propio y el 3 compite con el 2. Por eso ninguna plantilla de la regla lleva el
reloj A del nivel 3 a su horquilla (45–75), y ese objetivo pasó de ajuste a comprobación
después de medir la sensibilidad (regla 5): es una discrepancia de estructura, no de
plantilla.

**Para experimentar**

- Quita meses de `FECHAS_CAL` (por ejemplo, solo de abril a octubre, `FECHAS_CAL[3:10]`):
  ¿qué plantilla de médicos pasaría con esa ventana? ¿Representa al año?
- ¿Qué pasa con el pretriaje si quitas la enfermera de la tarde (`[3, 1, 1]`)? Pásala
  como lista de una plantilla a `evaluate_staffings`.

> **Lo que debes recordar:** se calibra a lo observado con horquillas, con una réplica por mes para que la media sea la del año, dos mandos con objetivos separados y en orden, y una semilla de comprobación que puede vetar la elección, no hacerla; un objetivo que ningún mando mueve (el nivel 3) pasa a comprobación después de medir la sensibilidad, y la comparación con el SAS es de forma, porque sus cifras incluyen menores y van por nivel asignado.

## 10. El coste del infratriaje

El cumplimiento observable (`manchester_compliance`) agrupa a cada paciente por su nivel
**asignado** y le aplica el máximo de ese nivel: es lo que puede medir el hospital. Con el
ground truth se puede medir también con el nivel **real**
(`true_level_compliance`): la misma regla, con el nivel real para agrupar y para elegir el
máximo. Un nivel 2 real asignado como 3 que ve al médico en 30 minutos cumple su 3 (60)
y no su 2 (10). La diferencia entre las dos cifras es **el coste del infratriaje medido en
tiempo**. Nunca entra en el log ni en los KPIs observables.

In [ ]:
por_asignado = manchester_compliance(tabla, CAL, DIAS)  # reloj C
por_real = true_level_compliance(tabla, verdad, CAL, DIAS)
fig, (ax_p, ax_d) = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(2, 6)
ax_p.bar(
    x - 0.2,
    100 * por_asignado.pooled_within[1:],
    width=0.4,
    color=OBSERVED_COLOR,
    label="por nivel asignado (observable)",
)
ax_p.bar(
    x + 0.2,
    100 * por_real.pooled_within[1:],
    width=0.4,
    color=THEORY_COLOR,
    label="por nivel real (ground truth)",
)
ax_p.axhline(90, color="k", lw=1, ls="--")
ax_p.set_ylim(0, 105)
ax_p.set_ylabel("% de pacientes dentro del máximo (reloj C)")
ax_p.set_title(f"Cumplimiento Manchester en {DIAS - CAL} días")
ax_d.bar(
    x - 0.2,
    por_asignado.out.sum(axis=0)[1:],
    width=0.4,
    color=OBSERVED_COLOR,
    label="por nivel asignado",
)
ax_d.bar(
    x + 0.2, por_real.out.sum(axis=0)[1:], width=0.4, color=THEORY_COLOR, label="por nivel real"
)
ax_d.set_ylabel(f"días fuera, de {DIAS - CAL}")
ax_d.set_title("Días con menos del 90 % dentro del máximo")
for ax in (ax_p, ax_d):
    ax.set_xticks(x, [f"nivel {k}" for k in x])
    ax.legend(fontsize=8, loc="lower right" if ax is ax_p else "upper left")
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print("Nivel  dentro (asignado)  dentro (real)  días fuera (asignado)  días fuera (real)")
for k in range(2, 6):
    dentro_a, dentro_r = (
        100 * por_asignado.pooled_within[k - 1],
        100 * por_real.pooled_within[k - 1],
    )
    fuera_a, fuera_r = por_asignado.out.sum(axis=0)[k - 1], por_real.out.sum(axis=0)[k - 1]
    print(f"{k:>5}  {dentro_a:16.1f}%  {dentro_r:12.1f}%  {fuera_a:21d}  {fuera_r:17d}")

In [ ]:
reloj_c = true_level_waits(tabla, verdad, "classified_to_doctor", CAL, DIAS)
print("Niveles 2 reales según su nivel asignado (reloj C):")
for asignado in (1, 2, 3):
    g = (reloj_c.true == 2) & (reloj_c.assigned == asignado)
    if g.any():
        c = reloj_c.values[g]
        print(
            f"  asignado {asignado}: {g.sum():5d} pacientes, C medio {c.mean():6.1f} min,"
            f" dentro de los 10 min {100 * np.mean(c <= 10):5.1f} %"
        )
infra = reloj_c.assigned > reloj_c.true
print(f"\nInfratriados: {infra.sum()} de {infra.size} ({100 * infra.mean():.1f} %)")

En la ejecución normal (89 días, reloj C), el cumplimiento por nivel real queda por debajo
del observable en los niveles urgentes: el nivel 2 pasa del 89,8 % al 88,4 % de pacientes
dentro del máximo y sus días fuera, de 34 a 42; el nivel 3, de 17 días fuera a 35. Los 766
niveles 2 reales asignados como 3 esperan 9,7 minutos de media desde la clasificación,
frente a 3,9 los bien clasificados, y solo el 82,8 % llega en 10 minutos. En el nivel 5
pasa lo contrario: muchos niveles 5 reales están asignados como 4, esperan menos de lo que
su máximo permite y, por nivel real, el nivel 5 queda fuera 40 días en lugar de 56. El
hospital no puede ver esa diferencia: solo el ground truth sabe quién estaba mal
clasificado.

**Para experimentar**

- Mira el nivel 5: ¿por qué por nivel real tiene más cumplimiento que por nivel asignado?
  (Pista: muchos niveles 5 reales están asignados como 4.)
- Repite con el reloj A (`measure="door_to_doctor"`). ¿Cambia la diferencia?

> **Lo que debes recordar:** el cumplimiento observable usa el nivel asignado; el mismo cálculo con el nivel real, que solo da el ground truth, mide el coste del infratriaje en tiempo: los urgentes infratriados esperan como su nivel asignado y su cumplimiento real cae.

## 11. Centro realista frente a centro cumplidor, con los mismos pacientes

El centro calibrado es **realista**: incumple a veces, como los hospitales reales. El
cumplidor (`config/centro_cumplidor.toml`) tiene la misma enfermería y los médicos de la
regla con el u más alto con el que cada nivel 2 a 5 queda en un 10 % de días fuera de
Manchester como mucho, con los días sumados en las réplicas: [14, 14, 6] frente a
[12, 11, 4]. Con la
misma semilla, cada paciente es el mismo en los dos centros (números aleatorios comunes):
llega a la misma hora, con el mismo motivo, la misma gravedad y el mismo nivel, y solo
cambian sus esperas.

In [ ]:
tabla_c, verdad_c = ejecuta(CUMPLIDOR, PERFILES, DIAS, SEMILLA_PRINCIPAL)
par = paired_waits(tabla, tabla_c, "classified_to_doctor", CAL * DIA, DIAS * DIA)
cumple_c = manchester_compliance(tabla_c, CAL, DIAS)
mismos = verdad.episodes == verdad_c.episodes
print(f"{len(par.episode_ids)} pacientes emparejados; el mismo ground truth: {mismos}")
fig, (ax_m, ax_f) = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(2, 6)
medias_b = [par.base[par.levels == k].mean() for k in x]
medias_c = [par.other[par.levels == k].mean() for k in x]
ax_m.bar(
    x - 0.2, medias_b, width=0.4, color=OBSERVED_COLOR, label=f"calibrado {list(CENTRO.doctors)}"
)
ax_m.bar(
    x + 0.2, medias_c, width=0.4, color=SECOND_COLOR, label=f"cumplidor {list(CUMPLIDOR.doctors)}"
)
for xi, (b, c) in enumerate(zip(medias_b, medias_c, strict=True)):
    ax_m.text(x[xi] - 0.2, b + 2, f"{b:.1f}", ha="center", fontsize=8)
    ax_m.text(x[xi] + 0.2, c + 2, f"{c:.1f}", ha="center", fontsize=8)
ax_m.set_ylabel("clasificación-médico medio, reloj C (min)")
ax_m.set_title("Los mismos pacientes en los dos centros")
ax_f.bar(
    x - 0.2, 100 * por_asignado.share_out[1:], width=0.4, color=OBSERVED_COLOR, label="calibrado"
)
ax_f.bar(x + 0.2, 100 * cumple_c.share_out[1:], width=0.4, color=SECOND_COLOR, label="cumplidor")
ax_f.axhline(10, color="k", lw=1, ls="--", label="10 % de días fuera")
ax_f.set_ylabel(f"% de días fuera de Manchester (de {DIAS - CAL})")
ax_f.set_title("Días fuera con el reloj C")
for ax in (ax_m, ax_f):
    ax.set_xticks(x, [f"nivel {k}" for k in x])
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print(
    "Nivel  C calibrado  C cumplidor  diferencia media  esperan menos  días fuera (cal. / cumpl.)"
)
for k in x:
    g = par.levels == k
    menos = 100 * np.mean(par.difference[g] < 0)
    fuera_b, fuera_c = 100 * por_asignado.share_out[k - 1], 100 * cumple_c.share_out[k - 1]
    print(
        f"{k:>5}  {par.base[g].mean():11.1f}  {par.other[g].mean():11.1f}"
        f"  {par.difference[g].mean():+16.1f}  {menos:12.1f}%  {fuera_b:9.1f} / {fuera_c:.1f} %"
    )

En la ejecución normal (89 días, 28.493 pacientes emparejados), con los mismos pacientes,
el cumplidor baja el reloj C medio del nivel 4 de 98,3 a 9,4 minutos y el del nivel 5, de
612 a 22; en los niveles 2 y 3 la diferencia es de 3 y 9 minutos. Los días fuera de
Manchester con C caen del 84 y el 63 % al 5,6 y el 9,0 % en los niveles 4 y 5, y a 0 en los
niveles 2 y 3. Para eso necesita 7 médicos más, sumando los tres turnos (34 frente a 27), y
su reloj A del año queda por debajo de la horquilla observada (15,65 minutos en el
registro): cumple, pero ya no se parece a un hospital español medio.

**Para experimentar**

- Empareja el reloj A (`"door_to_doctor"`) en lugar de C: ¿cambia el pretriaje entre los
  dos centros? ¿Por qué no puede cambiar?
- ¿Qué turno explica más diferencia? Separa `par.arrivals` por hora del día.

> **Lo que debes recordar:** el centro base es realista e incumple a veces, y el cumplidor es una variante con más médicos; con números aleatorios comunes se comparan los mismos pacientes, y la diferencia se concentra en los niveles 4 y 5, que cargan con la prioridad estricta.

## 12. Para experimentar

Tres cambios, cada uno con la misma semilla y 28 días:

- **Otro reparto objetivo: el del SAS en adultos.** El CMBD 2013 da el reparto por nivel
  con niños (0,93 / 6,37 / 30,23 / 40,11 / 22,36 %). Quitando los menores de cada nivel
  (2,65 / 9,60 / 10,38 / 23,78 / 35,90 %) queda 1,15 / 7,32 / 34,44 / 38,86 / 18,22 %:
  muchos más niveles 5 que en Emergencias 2013 (1,4 %). `with_target_mix` crea un perfil
  con ese reparto y **rederiva los umbrales en memoria** con `thresholds_for`, sin tocar
  `config/`. **El ruido y el sesgo se quedan como están**, calibrados para el reparto
  base, así que el infratriaje y el sobretriaje de la variante cambian.
- **Otro ruido de triaje**, con los umbrales escritos: una enfermería más insegura.
- **Otra plantilla de médicos.**

In [ ]:
REPARTO_EXP = sas.level_pct(sas.adult_level_counts())  # SAS en adultos, %
RUIDO_EXP = 1.5 * PERFILES.triage.noise  # con los umbrales escritos
MEDICOS_EXP = tuple(np.add(CENTRO.doctors, (1, 1, 0)).tolist())  # +1 de mañana y de tarde
SEMILLA_EXP = 2026

sas_adultos = with_target_mix(PERFILES, LLEGADAS, REPARTO_EXP)
ruidoso = dataclasses.replace(
    PERFILES, triage=dataclasses.replace(PERFILES.triage, noise=RUIDO_EXP)
)
variantes = {
    "base": (CENTRO, PERFILES),
    "reparto del SAS": (CENTRO, sas_adultos),
    f"ruido ×{RUIDO_EXP / PERFILES.triage.noise:g}".replace(".", ","): (CENTRO, ruidoso),
    f"médicos {list(MEDICOS_EXP)}": (dataclasses.replace(CENTRO, doctors=MEDICOS_EXP), PERFILES),
}
for nombre, perfiles in (("base", PERFILES), ("reparto del SAS", sas_adultos), ("ruido", ruidoso)):
    tt = perfiles.triage
    m = triage_matrix(severity_mixture(perfiles, LLEGADAS), tt.thresholds, tt.noise, tt.bias)
    inf, sob = triage_error_rates(m)
    asignado = np.round(100 * m.sum(axis=0), 2)
    print(
        f"{nombre:<16} umbrales {np.round(tt.thresholds, 3)}  asignado {asignado} %"
        f"  infra {100 * inf:.1f} %, sobre {100 * sob:.1f} %"
    )

In [ ]:
resultados = {}
for nombre, (centro, perfiles) in variantes.items():
    tabla_v, verdad_v = ejecuta(centro, perfiles, DIAS_EXP, SEMILLA_EXP)
    r = replica_values(tabla_v, OBJETIVOS, CAL, DIAS_EXP)
    real = true_level_compliance(tabla_v, verdad_v, CAL, DIAS_EXP)
    resultados[nombre] = (r, real)
print(
    f"{'variante':<20}{'pretriaje':>10}{'A 2-5':>8}{'A nivel 3':>10}{'% sobre M.':>11}"
    f"   días fuera C, niveles 2-5 (asignado)   dentro C por nivel real 2"
)
for nombre, (r, real) in resultados.items():
    v = r.targets
    print(
        f"{nombre:<20}{v['pretriaje_medio'].value:>10.1f}{v['tiempo_medico_niveles_2_5'].value:>8.1f}"
        f"{v['tiempo_medico_nivel_3'].value:>10.1f}{v['porcentaje_sobre_manchester'].value:>10.1f}%"
        f"   {' / '.join(f'{100 * x:4.0f}' for x in r.compliance_out[1:])} %"
        f"{100 * real.pooled_within[1]:>22.1f} %"
    )

fig, (ax_r, ax_k) = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(1, 6)
for i, (nombre, perfiles, color) in enumerate(
    (
        ("Emergencias 2013 (base)", PERFILES, OBSERVED_COLOR),
        ("SAS 2013 en adultos", sas_adultos, THEORY_COLOR),
    )
):
    tt = perfiles.triage
    ax_r.bar(
        x + (i - 0.5) * 0.4,
        100
        * assigned_shares(severity_mixture(perfiles, LLEGADAS), tt.thresholds, tt.noise, tt.bias),
        width=0.4,
        color=color,
        label=nombre,
    )
ax_r.set_xticks(x, [f"nivel {k}" for k in x])
ax_r.set_ylabel("% asignado (exacto, un año)")
ax_r.set_title("Dos repartos objetivo, cada uno con sus umbrales")
ax_r.legend(fontsize=8)
nombres_v = list(resultados)
ax_k.axhspan(35, 70, color=SECOND_COLOR, alpha=0.15, label="horquilla 35–70", zorder=0)
ax_k.bar(
    range(len(nombres_v)),
    [resultados[n][0].targets["tiempo_medico_niveles_2_5"].value for n in nombres_v],
    color=[OBSERVED_COLOR, THEORY_COLOR, "#7b5ea7", MUTED_COLOR],
    width=0.6,
    zorder=2,
)
ax_k.set_xticks(range(len(nombres_v)), nombres_v, fontsize=8)
ax_k.set_ylabel("llegada-médico medio, niveles 2 a 5 (min)")
ax_k.set_title(f"Reloj A con cada variante ({DIAS_EXP - CAL} días, semilla {SEMILLA_EXP})")
ax_k.legend(fontsize=8)
for ax in (ax_r, ax_k):
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

En la ejecución normal (26 días medidos, una sola ejecución con la semilla 2026), el centro
base da un reloj A de los niveles 2 a 5 de 141,7 minutos, el doble de su horquilla. Son las
primeras semanas de enero, y entre ellas está la del 19 al 25: cuatro días seguidos con más
llegadas de lo esperado (efecto diario de 1,03 a 1,18). Con 4 médicos de noche la cola de la
noche pasa al día siguiente y crece: a las 8:00 llega a haber 129 pacientes clasificados
esperando médico (el viernes 23), y la cola no se vacía hasta el viernes 30. **Cerca de la
saturación, los días malos no se compensan con los buenos: se acumulan**, y pagan los
niveles 4 y 5, que van al final de la cola. Es el enero de una plantilla calibrada para la
media del año (sección 9). No es efecto del arranque vacío: con el factor mensual neutro,
los primeros 14 días medidos dan lo mismo que los siguientes (registro de calibración, «El
reloj A según el horizonte»).

- **Reparto del SAS.** Los umbrales pasan a [2,547; 1,394; 0,037; −1,031] y el nivel 5
  asignado, del 1,4 % al 18,2 %. Como el ruido y el sesgo no se recalibran, el infratriaje
  sube al 14,4 % y el sobretriaje al 8,0 %. Con los mismos pacientes, el reloj A baja a
  74,9 minutos. La causa es la sala de resultados (decisión 45): hay menos niveles 2
  asignados, que retienen el box mientras esperan sus pruebas, y más 4 y 5, que lo dejan.
  Con los boxes más libres, la espera de box de los niveles 4 y 5 cae. **El nivel asignado
  decide quién libera el box**, aunque la gravedad de los pacientes sea la misma.
- **Ruido ×1,5.** Con los umbrales escritos, el reparto asignado se ensancha (más niveles 2
  y 5) y el infratriaje y el sobretriaje suben al 13,5 y al 10,7 %. Las medias casi no se
  mueven (reloj A de 144,6 minutos), pero el cumplimiento del nivel 2 por nivel real baja
  del 86,0 al 85,4 % con el reloj C, porque sube el infratriaje. Con tan
  pocos días y tan cerca de la saturación, los días fuera por nivel asignado cambian sobre
  todo con la semana mala, y no se pueden leer como un efecto del ruido.
- **Médicos [13, 12, 4].** Un médico más por la mañana y otro por la tarde bajan el reloj A
  a 68,9 minutos, dentro de la horquilla, y los días fuera de los niveles 2, 3 y 5; la
  noche sigue con 4 médicos.

Preguntas guía:

- **Reparto:** con el reparto del SAS, ¿qué nivel gana más pacientes? ¿Por qué baja la
  espera media de los niveles 2 a 5 aunque lleguen los mismos pacientes?
- **Ruido:** con más ruido, ¿cuánto cae el cumplimiento por nivel real del nivel 2?
  ¿Lo vería el hospital con su cumplimiento observable?
- **Plantilla:** ¿qué turno conviene reforzar? Prueba `(12, 11, 5)` y `(13, 11, 4)`.

> **Lo que debes recordar:** un reparto objetivo nuevo es un perfil nuevo, con sus umbrales rederivados en memoria; más ruido con los mismos umbrales es un shock que el hospital solo ve en parte, y cada cambio se prueba en segundos con la misma semilla.

## 13. Resumen

Lo que debes recordar de cada sección:

2. Cada atributo se sortea condicionado a los anteriores, en orden fijo y con un flujo por
   fuente; el log solo lleva lo que ve el hospital.
3. Un factor multiplica el peso; el reparto del catálogo es la media de un año, y los pesos
   base se derivan para reproducirlo.
4. Gumbel-max, con un solo peso que crece, solo cambia a los que pasan a esa categoría: el
   mínimo posible. La CDF inversa mueve a muchos más.
5. El nivel asignado es G + ε con los umbrales y el real, G; los errores se concentran en
   los niveles raros.
6. La puerta decide con el nivel asignado: el crítico infratriado espera como un 2.
7. Pruebas y duraciones siguen la gravedad real: el infratriado espera como su nivel
   asignado y consume como el real.
8. El cumplimiento se mide desde la clasificación (C); con el reloj A el nivel 2 no puede
   cumplir.
9. Se calibra a lo observado, con horquillas, la media del año (una réplica por mes) y
   dos mandos separados; la semilla de comprobación puede vetar, y con una plantilla para
   todo el año enero queda cerca de la saturación. El nivel 3 queda como comprobación por
   una discrepancia de estructura.
10. El cumplimiento por nivel real mide el coste del infratriaje en tiempo.
11. Con números aleatorios comunes se comparan los mismos pacientes en dos centros.
12. Un reparto objetivo nuevo es un perfil nuevo con sus umbrales; el ruido con los
    umbrales escritos es un shock.

Las cuatro frases que pide el hito:

- **Un motivo se mezcla con su contexto** multiplicando su peso base por un factor de
  cada rasgo del contexto (edad, franja, día y estación) y renormalizando; el sorteo es
  Gumbel-max, para que un escenario solo cambie a quien debe.
- **El nivel asignado no es el real** porque la enfermera clasifica G + ε: la prioridad
  usa el asignado, pero las pruebas y las duraciones, la gravedad real. Un infratriado
  espera de más y consume como su nivel real.
- **El cumplimiento se mide desde la clasificación** porque antes no hay nivel, y así se
  separa la cola de triaje (enfermería) de la de boxes y médicos.
- **Se calibra a lo observado con dos mandos de objetivos separados** porque un centro
  calibrado a la norma cumpliría por diseño, y porque la enfermería mueve el pretriaje y
  los médicos la espera desde la clasificación: cada mando es identificable con su
  objetivo.

In [ ]:
pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # en Linux, KB → MB
print(
    f"Tiempo total: {time.perf_counter() - INICIO:.0f} s; pico de memoria del kernel: {pico:.0f} MB"
)